# 03 — Extracción OCR Multimodal y Refactorización Estructurada (Agente 2)

**Objetivo:** Evaluar la transcripción óptica de los cuadrantes segmentados mediante el **Agente 2**:
1. Comparar la línea de base local (`Tesseract OCR`) contra la cascada VLM (`Ollama` / `NVIDIA Nemotron 3.5 Parse` + `LLM Refactorer`).
2. Inspeccionar la salida cruda (`LaTeX`/`Markdown`) frente al esquema JSON normalizado (`series`, `flechas`, `totales_planilla`).

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd

try:
    from tiro_arco.agente_1_clasificador_segmentador import AgenteClasificadorSegmentador
    from tiro_arco.agente_2_ocr_extractor import Agente2OCRExtractor
except ImportError:
    from agente_1_clasificador_segmentador_2 import AgenteClasificadorSegmentador
    from agente_2_ocr_extractor_2 import Agente2OCRExtractor

RAW_DIR = '../data/raw'
muestra = os.path.join(RAW_DIR, '1-AFPSF_Plani_TS-4-Arqueros.png')
if not os.path.exists(muestra):
    candidatos = [os.path.join(RAW_DIR, f) for f in os.listdir(RAW_DIR) if f.lower().endswith(('.png', '.jpg', '.pdf'))]
    muestra = sorted(candidatos)[0]

agente1 = AgenteClasificadorSegmentador(output_dir='/tmp/nb03_crops')
manifiesto_1 = agente1.procesar_documento(muestra)

agente2 = Agente2OCRExtractor(output_dir='/tmp/nb03_ocr')
print('Cadena de modelos configurada:', agente2.model_chain)

manifiesto_2 = agente2.procesar_manifiesto_agente1(manifiesto_1)

resumen = []
for t in manifiesto_2.get('transcripciones', []):
    tr = t.get('transcripcion', {})
    resumen.append({
        'pagina': t.get('pagina'),
        'target_id': t.get('target_id'),
        'arquero': tr.get('arquero'),
        'club': tr.get('club'),
        'categoria': tr.get('categoria'),
        'series_detectadas': len(tr.get('series', [])),
        'total_calculado': tr.get('totales_planilla_calculados', {}).get('total_puntos', 0),
        'modelo_usado': tr.get('modelo_usado'),
        'tiempo_seg': tr.get('tiempo_inferencia_seg', 0)
    })
display(pd.DataFrame(resumen))

In [ ]:
tareas_activas = [t for t in manifiesto_1['tareas_agente2'] if not t.get('es_pagina_informacion') and not t.get('es_cuadrante_vacio')]
if tareas_activas:
    tarea_demo = tareas_activas[0]
    res_tess = agente2._fallback_tesseract_ocr(
        tarea_demo['ruta_recorte'],
        tarea_demo['tipo_juego'],
        target_id_default=tarea_demo['target_id']
    )
    res_vlm = next((
        t['transcripcion'] for t in manifiesto_2['transcripciones']
        if t['pagina'] == tarea_demo['pagina'] and t['target_id'] == tarea_demo['target_id']
    ), {})

    comp = pd.DataFrame([
        {
            'Motor': 'Tesseract OCR (Baseline Local)',
            'Arquero': res_tess.get('arquero'),
            'Club': res_tess.get('club'),
            'Categoría': res_tess.get('categoria'),
            'Series': len(res_tess.get('series', [])),
            'Total Calc': res_tess.get('totales_planilla_calculados', {}).get('total_puntos', 0)
        },
        {
            'Motor': res_vlm.get('modelo_usado', 'Pipeline VLM'),
            'Arquero': res_vlm.get('arquero'),
            'Club': res_vlm.get('club'),
            'Categoría': res_vlm.get('categoria'),
            'Series': len(res_vlm.get('series', [])),
            'Total Calc': res_vlm.get('totales_planilla_calculados', {}).get('total_puntos', 0)
        }
    ])
    display(comp)
    print('--- Muestra JSON Estructurado (Serie 1 a 3) ---')
    print(json.dumps(res_vlm.get('series', [])[:3], indent=2, ensure_ascii=False))